In [ ]:
import json, re, random, os, sys
from dotenv import load_dotenv, find_dotenv
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from typing import List, Dict, Any
import statistics
from IPython.display import display, HTML

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

def find_project_root() -> Path:
    p = Path.cwd()
    markers = {".git", "pyproject.toml", ".env"}
    for up in [p, *p.parents]:
        if any((up / m).exists() for m in markers):
            return up
    return p

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

env_path = find_dotenv(filename=".env", usecwd=True) or str(PROJECT_ROOT / ".env")
print("Loaded .env from:", env_path)
load_dotenv(env_path, override=False)

from ingest.constants import SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY, STORAGE_BUCKET, PDF_FILENAME, HUGGING_FACE_HUB_TOKEN
from ingest.chunk_import import slug, norm_title, extract_text_pages, windows, flatten, descendants

# Functions to test
from ingest.chunk_import import fetch_pdf_from_storage, build_toc, chapter_intervals, chunk_sections_with_hierarchy
from ingest.embedding_import import generate_embeddings, validate_embeddings

In [ ]:
'''Chunk Display'''

# Fetch PDF
pdf_bytes = fetch_pdf_from_storage(
    SUPABASE_URL,
    SUPABASE_SERVICE_ROLE_KEY,
    STORAGE_BUCKET,
    PDF_FILENAME
)

doc_key, toc, page_count = build_toc(pdf_bytes)
chunks = chunk_sections_with_hierarchy(pdf_bytes, toc)

def generate_embeddings(
    texts: List[str],
    model_name: str = "BAAI/bge-small-en-v1.5",
    batch_size: int = 64,
    normalize: bool = True,
    show_progress: bool = True
) -> np.ndarray:

    print(f"🤖 Loading model: {model_name}")
    model = SentenceTransformer(model_name)
    
    print(f"⚙️  Generating embeddings for {len(texts)} texts...")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=show_progress,
        normalize_embeddings=normalize,
        convert_to_numpy=True  # Explicit numpy conversion
    )
    
    print(f"✅ Generated embeddings with shape: {embeddings.shape}")
    return embeddings

# Build comprehensive DataFrame
records = []
for i, chunk in enumerate(chunks):
    records.append({
        'chunk_id': i,
        'doc_key': doc_key,
        'section_id': chunk['section_id'],
        'section_title': chunk['section_title'],
        'chunk_seq': chunk['chunk_seq'],
        'level': chunk['level'],
        'hierarchy': ' → '.join(chunk.get('hierarchy', [])),
        'page_start': chunk['page_start'],
        'page_end': chunk['page_end'],
        'page_range': f"{chunk['page_start']}-{chunk['page_end']}",
        'text_length': len(chunk['text']),
        'text_preview': chunk['text'][:150] + '...' if len(chunk['text']) > 150 else chunk['text'],
        'embedding_dim': embeddings.shape[1],
        'embedding_norm': norms[i],
        'embedding_mean': embeddings[i].mean(),
        'embedding_std': embeddings[i].std(),
        'embedding_preview': str(embeddings[i][:5].round(4).tolist()),
        # Store full data (hidden from display)
        'full_text': chunk['text'],
        'full_hierarchy': chunk.get('hierarchy', []),
        'full_embedding': embeddings[i]
    })

df = pd.DataFrame(records)

print(f"✅ Created DataFrame with {len(df)} rows and {len(df.columns)} columns\n")

# Display overview
print("📋 DataFrame Overview:")
display(df[['chunk_id', 'section_title', 'level', 'hierarchy', 'text_length', 
            'embedding_dim', 'embedding_norm']].head(10))

In [ ]:
# ============ GENERATE EMBEDDINGS ============
print("\n" + "="*80)
print("GENERATING EMBEDDINGS")
print("="*80 + "\n")

texts = [chunk["text"] for chunk in chunks]
embeddings = generate_embeddings(texts)

# ============ ANALYZE EMBEDDINGS ============
print("\n" + "="*80)
print("EMBEDDING ANALYSIS")
print("="*80 + "\n")

# 1. Basic Statistics
print("📊 Basic Statistics:")
print(f"   Total chunks: {len(chunks)}")
print(f"   Total embeddings: {len(embeddings)}")
print(f"   Embedding dimension: {embeddings.shape[1]}")
print(f"   Memory usage: {embeddings.nbytes / 1024 / 1024:.2f} MB")
print(f"   Data type: {embeddings.dtype}")

# 2. Normalization Check (Critical for cosine similarity)
print("\n📏 Normalization Check:")
norms = np.linalg.norm(embeddings, axis=1)
print(f"   Min norm: {norms.min():.6f}")
print(f"   Max norm: {norms.max():.6f}")
print(f"   Mean norm: {norms.mean():.6f}")
print(f"   Std norm: {norms.std():.6f}")
print(f"   All normalized? {np.allclose(norms, 1.0, atol=1e-5)}")

# 3. Quality Checks
print("\n🔍 Quality Checks:")
has_nan = np.isnan(embeddings).any()
has_inf = np.isinf(embeddings).any()
zero_vectors = np.all(embeddings == 0, axis=1).sum()

print(f"   Contains NaN: {has_nan}")
print(f"   Contains Inf: {has_inf}")
print(f"   Zero vectors: {zero_vectors}")

if has_nan or has_inf or zero_vectors > 0:
    print("\n   ⚠️  WARNING: Found quality issues!")
else:
    print("\n   ✅ All quality checks passed!")

# 4. Similarity Analysis (Check for duplicates and diversity)
print("\n🔗 Similarity Analysis:")
# Compute pairwise similarities (sample if too large)
sample_size = min(100, len(embeddings))
sample_embeddings = embeddings[:sample_size]
similarity_matrix = sample_embeddings @ sample_embeddings.T
np.fill_diagonal(similarity_matrix, 0)  # Ignore self-similarity

print(f"   Sample size: {sample_size} chunks")
print(f"   Min similarity: {similarity_matrix.min():.4f}")
print(f"   Max similarity: {similarity_matrix.max():.4f}")
print(f"   Mean similarity: {similarity_matrix.mean():.4f}")
print(f"   Median similarity: {np.median(similarity_matrix):.4f}")

# Check for near-duplicates
near_duplicates = np.sum(similarity_matrix > 0.95)
print(f"   Near-duplicates (>0.95): {near_duplicates}")

# 5. Embedding Distribution Analysis
print("\n📈 Embedding Value Distribution:")
print(f"   Min value: {embeddings.min():.6f}")
print(f"   Max value: {embeddings.max():.6f}")
print(f"   Mean value: {embeddings.mean():.6f}")
print(f"   Std value: {embeddings.std():.6f}")

# ============ CREATE COMBINED DATAFRAME ============
print("\n" + "="*80)
print("CREATING COMBINED CHUNKS + EMBEDDINGS DATAFRAME")
print("="*80 + "\n")


In [ ]:
# ============ TEXT LENGTH ANALYSIS ============
print("\n" + "="*80)
print("TEXT LENGTH ANALYSIS")
print("="*80 + "\n")

print(f"Text Length Statistics:")
print(f"   Min: {df['text_length'].min()} chars")
print(f"   Max: {df['text_length'].max()} chars")
print(f"   Mean: {df['text_length'].mean():.0f} chars")
print(f"   Median: {df['text_length'].median():.0f} chars")
print(f"   Q1: {df['text_length'].quantile(0.25):.0f} chars")
print(f"   Q3: {df['text_length'].quantile(0.75):.0f} chars")

# Check if any chunks are too short or too long
too_short = (df['text_length'] < 100).sum()
too_long = (df['text_length'] > 5000).sum()
print(f"\n   Chunks < 100 chars: {too_short}")
print(f"   Chunks > 5000 chars: {too_long}")

# ============ LEVEL DISTRIBUTION ============
print("\n" + "="*80)
print("HIERARCHY LEVEL DISTRIBUTION")
print("="*80 + "\n")

level_dist = df['level'].value_counts().sort_index()
print("Chunks by heading level:")
for level, count in level_dist.items():
    pct = (count / len(df)) * 100
    print(f"   H{level}: {count:4d} chunks ({pct:5.1f}%)")

# ============ SAMPLE CHUNK + EMBEDDING DISPLAY ============
print("\n" + "="*80)
print("SAMPLE CHUNK WITH EMBEDDING")
print("="*80 + "\n")

sample_idx = 0
sample_chunk = chunks[sample_idx]
sample_embedding = embeddings[sample_idx]

print(f"Chunk ID: {sample_idx}")
print(f"Section: {sample_chunk['section_title']}")
print(f"Hierarchy: {' → '.join(sample_chunk.get('hierarchy', []))}")
print(f"Level: H{sample_chunk['level']}")
print(f"Pages: {sample_chunk['page_start']}-{sample_chunk['page_end']}")
print(f"Text length: {len(sample_chunk['text'])} chars")
print(f"\nText preview:")
print("-" * 80)
print(sample_chunk['text'][:500])
print("...")
print("-" * 80)
print(f"\nEmbedding info:")
print(f"   Shape: {sample_embedding.shape}")
print(f"   Norm: {np.linalg.norm(sample_embedding):.6f}")
print(f"   First 10 dimensions: {sample_embedding[:10].round(4).tolist()}")

# ============ SIMILARITY HEATMAP (Optional visualization) ============
print("\n" + "="*80)
print("TOP SIMILAR CHUNK PAIRS")
print("="*80 + "\n")

# Find most similar chunks (excluding self-similarity)
sample_size = min(50, len(embeddings))
sample_emb = embeddings[:sample_size]
sim_matrix = sample_emb @ sample_emb.T
np.fill_diagonal(sim_matrix, -1)  # Exclude self

# Get top 5 most similar pairs
top_pairs = []
for i in range(sample_size):
    for j in range(i+1, sample_size):
        top_pairs.append((i, j, sim_matrix[i, j]))

top_pairs.sort(key=lambda x: x[2], reverse=True)

print("Top 5 most similar chunk pairs:")
for i, j, sim in top_pairs[:5]:
    print(f"\n   Chunks #{i} ↔ #{j} (similarity: {sim:.4f})")
    print(f"   [{chunks[i]['section_title']}]")
    print(f"   [{chunks[j]['section_title']}]")

# ============ PREPARE FOR SUPABASE INGESTION ============
print("\n" + "="*80)
print("PREPARING DATA FOR SUPABASE")
print("="*80 + "\n")

print("✅ Embeddings are ready for ingestion!")
print(f"\nRequired fields for Supabase 'chunks' table:")
print(f"   - chunk_id (string)")
print(f"   - doc_key (string)")
print(f"   - section_id (string)")
print(f"   - section_title (string)")
print(f"   - chunk_seq (integer)")
print(f"   - level (integer)")
print(f"   - hierarchy (text[] or jsonb)")
print(f"   - page_start (integer)")
print(f"   - page_end (integer)")
print(f"   - text (text)")
print(f"   - embedding (vector(384)) ← IMPORTANT!")
print(f"\n   💡 Make sure your Supabase table has a 'vector(384)' column for embeddings")

# Show sample record that will be inserted
print(f"\n📄 Sample record structure:")
sample_record = {
    "chunk_id": f"{doc_key}::{chunks[0]['section_id']}::c{chunks[0]['chunk_seq']:06d}",
    "doc_key": doc_key,
    "section_id": chunks[0]['section_id'],
    "section_title": chunks[0]['section_title'],
    "chunk_seq": chunks[0]['chunk_seq'],
    "level": chunks[0]['level'],
    "hierarchy": chunks[0].get('hierarchy', []),
    "page_start": chunks[0]['page_start'],
    "page_end": chunks[0]['page_end'],
    "text": chunks[0]['text'][:100] + "...",
    "embedding": embeddings[0][:5].tolist()  # Show first 5 dims only
}

import json
print(json.dumps(sample_record, indent=2, ensure_ascii=False))

print("\n✅ Analysis complete! Ready to proceed with ingestion.")